In [3]:
# ==============================================================================
# 🧪 PLAYGROUND INTERAKTIF (BISA DIJALANKAN LANGSUNG - 1 KLIK SAJA!)
# ==============================================================================
# Cell ini otomatis memuat model & embedding tersimpan dari disk jika belum ada di memori.
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer, util

# 1. LOAD MODEL TERSIMPAN (Hanya berjalan 1x saat pertama kali cell dijalankan)
ARTIFACTS_DIR = Path.cwd() / "output_artifacts"
FINETUNED_MODEL_DIR = ARTIFACTS_DIR / "finetuned_sbert"
METADATA_FILE = ARTIFACTS_DIR / "corpus_metadata.json"
EMBEDDINGS_FILE = ARTIFACTS_DIR / "corpus_embeddings_finetuned.npy"

if 'model_inferensi' not in globals():
    print("[AUTO-LOAD] Memuat model Fine-Tuned dari folder output_artifacts/...")
    model_inferensi = SentenceTransformer(str(FINETUNED_MODEL_DIR), device="cpu")
    with open(METADATA_FILE, "r", encoding="utf-8") as f:
        df_corpus_inferensi = pd.DataFrame(json.load(f))
    corpus_embeddings_inferensi = torch.tensor(np.load(EMBEDDINGS_FILE))
    print(f"[AUTO-LOAD] Model & {len(df_corpus_inferensi)} korpus materi siap digunakan!\n")

# 2. DEFINISI FUNGSI INFERENSI
def uji_pertanyaan_baru(query_teks: str, top_k: int = 3) -> dict:
    q_emb = model_inferensi.encode(query_teks, convert_to_tensor=True)
    cos_scores = util.cos_sim(q_emb, corpus_embeddings_inferensi)[0]
    top_results = torch.topk(cos_scores, k=min(top_k, len(df_corpus_inferensi)))

    print("=" * 72)
    print(f'INPUT PERTANYAAN BARU: "{query_teks}"')
    print("=" * 72)

    top_doc = None
    for idx, (score, doc_idx) in enumerate(zip(top_results.values, top_results.indices)):
        doc = df_corpus_inferensi.iloc[int(doc_idx)]
        if idx == 0:
            top_doc = doc
        print(f"Peringkat #{idx+1} [Confidence Score: {float(score):.4f} / {float(score)*100:.1f}%]")
        print(f"  • Bab     : {doc['chapter_title']} (Slug: {doc['chapter_slug']})")
        print(f"  • Episode : {doc['episode_title']}")
        print(f"  • Konsep  : {doc['concept']}")
        print(f"  • Cuplikan: \"{doc['passage_text'][:120]}...\"")
        print("-" * 72)

    speech = (
        f"Halo pembelajar! Pertanyaanmu berkaitan dengan {top_doc['chapter_title']}, "
        f"khususnya pada {top_doc['episode_title']}. "
        f"Ingat kembali konsep: {top_doc['concept']}. "
        f"Cobalah bereksperimen di editor kode tanpa melihat contekan!"
    )
    print("🔊 RESPONS SUARA SOKRATIK (UNTUK ESP32 TTS):\n" + f'"{speech}"')
    print("=" * 72 + "\n")

    return {
        "query": query_teks,
        "top_chapter": top_doc["chapter_title"],
        "top_episode": top_doc["episode_title"],
        "speech_esp32": speech,
        "confidence_score": float(top_results.values[0])
    }

# ==============================================================================
# 3. KETIK PERTANYAAN BARU DI BAWAH INI DAN JALANKAN:
# ==============================================================================
pertanyaan_bebas = "what is meant by a loop?"

hasil_uji = uji_pertanyaan_baru(pertanyaan_bebas, top_k=3)


INPUT PERTANYAAN BARU: "what is meant by a loop?"
Peringkat #1 [Confidence Score: 0.7063 / 70.6%]
  • Bab     : Perulangan for & while (Slug: perulangan)
  • Episode : Episode 3: Kontrol Alur Break, Continue & Else
  • Konsep  : Menghentikan perulangan (break), melompati iterasi (continue), dan klausa else perulangan
  • Cuplikan: "Loop for melakukan penugasan nilai ke variabel penampung iterasi pada setiap langkah putaran loop:..."
------------------------------------------------------------------------
Peringkat #2 [Confidence Score: 0.6938 / 69.4%]
  • Bab     : Perulangan for & while (Slug: perulangan)
  • Episode : Episode 3: Kontrol Alur Break, Continue & Else
  • Konsep  : Menghentikan perulangan (break), melompati iterasi (continue), dan klausa else perulangan
  • Cuplikan: "Cara mudah memahami klausa else pada loop adalah dengan membayangkannya berpasangan dengan pernyataan if di dalam loop. ..."
------------------------------------------------------------------------
Peringka